In [1]:
!pip install spacy nltk gnews sentence-transformers transformers torch
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
import spacy
import nltk
from nltk.corpus import wordnet
from gnews import GNews
from sentence_transformers import CrossEncoder
from transformers import pipeline
import itertools
import warnings

# Suppress Hugging Face/Transformer warnings for a cleaner console
warnings.filterwarnings("ignore")

class RobustMisinformationVerifier:
    def __init__(self):
        print("\nLoading AI Models (this takes a moment on the first run)...")
        nltk.download('wordnet', quiet=True)

        self.nlp = spacy.load("en_core_web_sm")
        self.cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        self.stance_model = pipeline("text-classification", model="FacebookAI/roberta-large-mnli")
        self.google_news = GNews(language='en', country='US', max_results=10)

        print("Models loaded successfully!\n")

    def get_synonyms(self, word, pos):
        synonyms = {word}
        wn_pos = wordnet.NOUN if pos in ["NOUN", "PROPN"] else wordnet.VERB if pos == "VERB" else None

        if wn_pos:
            for syn in wordnet.synsets(word, pos=wn_pos):
                for lemma in syn.lemmas():
                    clean_syn = lemma.name().replace('_', ' ')
                    synonyms.add(clean_syn)
                    if len(synonyms) >= 3:
                        break
                if len(synonyms) >= 3:
                    break
        return list(synonyms)

    def divide_nouns_and_expand_synonyms(self, claim):
        doc = self.nlp(claim)

        nouns = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"] and not token.is_stop]
        verbs = [token.text for token in doc if token.pos_ == "VERB" and not token.is_stop]

        noun_groups = []
        if nouns:
            for i in range(1, min(3, len(nouns) + 1)):
                for combo in itertools.combinations(nouns, i):
                    noun_groups.append(" ".join(combo))
        else:
            noun_groups = [claim]

        queries = set()
        for n_group in noun_groups[:4]:
            if verbs:
                for v in verbs:
                    for syn_v in self.get_synonyms(v, "VERB"):
                        queries.add(f"{n_group} {syn_v}")
            else:
                queries.add(n_group)

        return list(queries)[:8]

    def gather_min_articles(self, queries, min_articles=5):
        articles_pool = []
        seen_titles = set()

        print(f"Executing search across {len(queries)} query variations...")
        for q in queries:
            try:
                results = self.google_news.get_news(q)
                for article in results:
                    title = article.get('title', '')
                    if title and title not in seen_titles:
                        seen_titles.add(title)
                        articles_pool.append(article)
            except Exception:
                continue

            if len(articles_pool) >= 15:
                break

        return articles_pool

    def get_core_subject(self, claim):
        """Uses spaCy dependency parsing to find the grammatical subject of the claim."""
        doc = self.nlp(claim)
        subject_token = None

        # 1. Look for the syntactic subject (nsubj, csubj, etc.)
        for token in doc:
            if "subj" in token.dep_:
                subject_token = token
                break

        # 2. Fallback to the first noun if no clear subject exists
        if not subject_token:
            for token in doc:
                if token.pos_ in ["NOUN", "PROPN"]:
                    subject_token = token
                    break

        if not subject_token:
            return set()

        # Build a strict filter pool containing the subject and its synonyms
        valid_subjects = {subject_token.text.lower(), subject_token.lemma_.lower()}
        valid_subjects.update([s.lower() for s in self.get_synonyms(subject_token.text, subject_token.pos_)])

        return valid_subjects

    def extract_best_evidence_from_pool(self, claim, articles, min_required=5):
        if len(articles) < min_required:
            print(f"⚠️ Note: Only collected {len(articles)} distinct articles (Target: {min_required}).")
        else:
            print(f"Retrieved {len(articles)} distinct articles for analysis.")

        # Identify the exact subject we MUST find in the evidence
        core_subjects = self.get_core_subject(claim)
        print(f"Filtering out news that doesn't mention: {core_subjects}")

        valid_sentences = []
        for art in articles:
            text_block = f"{art.get('title', '')}. {art.get('description', '')}"
            sents = [s.strip() for s in text_block.split('. ') if len(s.strip()) > 10]

            for sent in sents:
                sent_lower = sent.lower()
                # GUARDRAIL: The sentence MUST contain the core subject or a direct synonym
                if core_subjects and not any(sub in sent_lower for sub in core_subjects):
                    continue
                valid_sentences.append(sent)

        if not valid_sentences:
            return None

        sentence_pairs = [[claim, sent] for sent in valid_sentences]
        scores = self.cross_encoder.predict(sentence_pairs)

        best_idx = scores.argmax()
        return valid_sentences[best_idx]

    def check_for_speculation(self, sentence):
        text = sentence.lower()
        hedge_words = [
            "wants", "reportedly", "according to", "rumor", "rumours",
            "speculation", "linked", "expected", "could", "interested in",
            "eyeing", "target", "poised", "talks", "agreed terms", "likely"
        ]

        for word in hedge_words:
            if word in text:
                return True
        return False

    def classify_claim(self, claim, evidence_sentence, total_articles):
        if not evidence_sentence or total_articles == 0:
            return {
                "verdict": "⚠️ UNVERIFIED (No news found explicitly matching the subject)",
                "confidence": 0.0,
                "evidence": None,
                "articles_compared": total_articles
            }

        input_text = f"{evidence_sentence} </s></s> {claim}"

        result = self.stance_model(input_text)[0]
        raw_label = result['label'].upper()
        confidence = round(result['score'] * 100, 2)

        if self.check_for_speculation(evidence_sentence):
            return {
                "verdict": "❌ MISINFORMATION (Premature Fact / Speculation Found)",
                "confidence": 0.0,
                "evidence": evidence_sentence,
                "articles_compared": total_articles
            }

        if "CONTRADICTION" in raw_label or "LABEL_0" in raw_label:
            verdict = "❌ MISINFORMATION (Contradicted by News)"
        elif "ENTAILMENT" in raw_label or "LABEL_2" in raw_label:
            verdict = "✅ VERIFIED (Supported by News)"
        else:
            verdict = "⚠️ UNVERIFIED (News context is neutral or inconclusive)"

        return {
            "verdict": verdict,
            "confidence": confidence,
            "evidence": evidence_sentence,
            "articles_compared": total_articles
        }

    def verify(self, claim):
        print("\n" + "="*70)
        print(f"CLAIM TO VERIFY: \"{claim}\"")
        print("="*70)

        queries = self.divide_nouns_and_expand_synonyms(claim)
        articles = self.gather_min_articles(queries, min_articles=5)

        print("Analyzing sentences for exact factual resolution...")
        best_evidence = self.extract_best_evidence_from_pool(claim, articles, min_required=5)

        result = self.classify_claim(claim, best_evidence, len(articles))

        print("\n" + "="*70)
        print("VERIFICATION SUMMARY")
        print("="*70)
        print(f"VERDICT:           {result['verdict']}")
        print(f"CONFIDENCE:        {result['confidence']}%")
        print(f"ARTICLES COMPARED: {result['articles_compared']}")

        if result['evidence']:
            print(f"BEST EVIDENCE:     \"{result['evidence']}\"")
        print("="*70 + "\n")


if __name__ == "__main__":
    verifier = RobustMisinformationVerifier()

    print("Welcome to the Offline NLP Misinformation Verifier.")
    print("Type 'exit' or 'quit' to stop the program.\n")

    while True:
        user_claim = input("Enter a news claim to verify: ").strip()

        if user_claim.lower() in ['exit', 'quit']:
            print("Exiting...")
            break

        if user_claim:
            verifier.verify(user_claim)


Loading AI Models (this takes a moment on the first run)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully!

Welcome to the Offline NLP Misinformation Verifier.
Type 'exit' or 'quit' to stop the program.

Enter a news claim to verify: argentina won worldcup 2026

CLAIM TO VERIFY: "argentina won worldcup 2026"
Executing search across 3 query variations...
Analyzing sentences for exact factual resolution...
Retrieved 18 distinct articles for analysis.
Filtering out news that doesn't mention: {'argentina', 'argentine republic'}

VERIFICATION SUMMARY
VERDICT:           ❌ MISINFORMATION (Contradicted by News)
CONFIDENCE:        98.75%
ARTICLES COMPARED: 18
BEST EVIDENCE:     "Spain suffocates Lionel Messi and Argentina, wins World Cup on Ferran Torres' extra-time winner  Yahoo SportsSpain wins 2026 FIFA World Cup with grueling 1-0 victory over Argentina  CBS NewsSpain edge Argentina after extra time to win 2nd World Cup title  ESPN"

Enter a news claim to verify: olise to move to real madrid

CLAIM TO VERIFY: "olise to move to real madrid"
Executing search across 3 que

KeyboardInterrupt: Interrupted by user